# Bayesian Optimization for Black-Box Functions (Submission 10)

This notebook implements  **"The Final Strike"** Bayesian Optimization pipeline configured for **Round 10 Queries** across all 8 independent black-box target functions.

### Core Strategic Configurations for Round 10:
* **Maximum Acquisition Depth Strategy (300 Multi-Starts):** To ensure maximum search resolution and fully eliminate non-convex acquisition surface traps, local optimization climbs are elevated to **300 random multi-starts** per function.
* **"The Final Strike" $\xi$-Mapping Architecture:**
  * **$\xi = 0.5000$ (Max Discovery Exploration):** Preserved for Channels 1, 3, 4, and 6 as a persistent wide-area diagnostic force to attempt last-mile global breakthroughs.
  * **$\xi = 0.1000$ (Noisy Landscape Scope):** Retained for Channel 2 to sustain a broad search profile over the stochastic process variance.
  * **$\xi = 0.0001$ (Hyper-Refined Convergence Recovery):** Dropped back down to 0.0001 for Channel 5 to hyper-refine the localized region now that the recovery phase has safely consolidated the area.
  * **$\xi = 0.00001$ (Surgical Peak Climbing):** Applied to Channel 7 for high-precision micro-stepping up toward the 2.55 landscape peak apex.
  * **$\xi = 0.0000$ (Pure Maximization Exploitation):** Dedicated to Channel 8, focusing all 300 randomized restarts entirely on pure localized gradient-ascent to hit the 10.0 target boundary exactly.

In [ ]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm

# Enforce inline visualization support
%matplotlib inline 

print("Environment initialized. Round 10 Final Strike pipeline standing by.")

## Definition of Object-Oriented Bayesian Optimizer

In [ ]:
class BayesianOptimizer:
    def __init__(self, is_noisy=False):
        """
        Initializes noise variance mappings.
        """
        self.alpha = 1e-2 if is_noisy else 1e-6
        self.model = None
        self.X = None
        self.Y = None
        self.dim = None

    def load_and_fit(self, func_dir):
        """
        Loads cumulative array multi-round datasets and fits a precise GPR model.
        """
        X = np.load(os.path.join(func_dir, "initial_inputs.npy"))
        Y = np.load(os.path.join(func_dir, "initial_outputs.npy"))
        self.X, self.Y, self.dim = X, Y, X.shape[1]
        
        kernel = ConstantKernel(1.0) * Matern(
            length_scale=np.ones(self.dim), 
            nu=2.5
        )
        
        self.model = GaussianProcessRegressor(
            kernel=kernel, 
            alpha=self.alpha,
            normalize_y=True, 
            n_restarts_optimizer=50 
        )
        self.model.fit(self.X, self.Y)

    def expected_improvement(self, x, xi):
        """
        Computes Expected Improvement (EI) surfaces for specified jitter configurations.
        """
        x = x.reshape(-1, self.dim)
        mu, sigma = self.model.predict(x, return_std=True)
        mu_sample_opt = np.max(self.Y)

        with np.errstate(divide='warn'):
            imp = mu - mu_sample_opt - xi
            Z = imp / sigma
            ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
            ei[sigma <= 0.0] = 0.0
        return ei

    def propose_next_point(self, xi):
        """
        Executes 300 multi-start restarts across the L-BFGS-B optimizer bound space.
        """
        best_x = None
        max_ei = -1
        # Increased to 300 restarts for the absolute maximum search depth
        for _ in range(300):
            x0 = np.random.uniform(0.0, 1.0, self.dim)
            res = minimize(
                lambda x: -self.expected_improvement(x, xi),
                x0,
                bounds=[(0.0, 1.0)] * self.dim,
                method='L-BFGS-B'
            )
            if -res.fun > max_ei:
                max_ei = -res.fun
                best_x = res.x
        return best_x

## Sequential Execution Optimization Pipeline

In [ ]:
output_file = "submission_10_results.txt"

# xi_map Logic: "The Final Strike"
# F1, F3, F4, F6: Maintaining max exploration (0.5) as a final chance for discovery.
# F5: Lowered to 0.0001. Now that we've recovered the region, we hyper-refine for the peak.
# F7: Lowered to 0.00001 for surgical climbing of the 2.55 peak.
# F8: Pure exploitation (0.0) + 300 restarts to hit the 10.0 target exactly.
xi_map = {
    1: 0.5000, 
    2: 0.1000, 
    3: 0.5000, 
    4: 0.5000, 
    5: 0.0001, 
    6: 0.5000, 
    7: 0.00001, 
    8: 0.0000  
}

print("Generating Submission 10: The Final Strike...")
print("-" * 55)

notebook_results = {}

for i in range(1, 8 + 1):
    func_dir = f"function_{i}"
    
    if not os.path.exists(func_dir):
        print(f"Warning: Functional target context '{func_dir}' missing from directory path.")
        continue
        
    optimizer = BayesianOptimizer(is_noisy=(i == 2))
    optimizer.load_and_fit(func_dir)
    
    current_xi = xi_map[i]
    next_point = optimizer.propose_next_point(xi=current_xi)
    formatted_point = "-".join([f"{val:.6f}" for val in next_point])
    
    notebook_results[f"Function {i}"] = (formatted_point, current_xi)
    print(f"Function {i}: {formatted_point} (xi={current_xi})")

# Write finalized coordinates to target result manifest
with open(output_file, "w") as f:
    for i in range(1, 9):
        key = f"Function {i}"
        if key in notebook_results:
            f.write(f"{key}: {notebook_results[key][0]}\n")

print("-" * 55)
print(f"Submission 10 successfully written onto file: {output_file}")

### Summary Evaluation Matrix

In [ ]:
print(f"| {'Target Channel':15} | {'Exploration (xi)':18} | {'Proposed Query Coordinates (Round 10)':55} |")
print(f"| {'-'*15} | {'-'*18} | {'-'*55} |")
for func, (coords, xi_val) in notebook_results.items():
    print(f"| {func:15} | {xi_val:<18} | {coords:55} |")